In [ ]:
import random
import string
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests

SOURCES = {
    "hanh_chinh_vn": {
        "url": "https://provinces.open-api.vn/api/v2/",
        "mo_ta": "API công khai lấy danh sách tỉnh/thành, quận/huyện, phường/xã Việt Nam. Dùng làm nguồn địa giới.",
        "cach_dung": "requests.get(url).json()"
    },
    "thegioididong": {
        "url": "https://www.thegioididong.com/",
        "mo_ta": "Tham khảo nhóm hàng điện thoại, laptop, phụ kiện, thiết bị điện tử và mặt bằng giá bán lẻ.",
        "cach_dung": "Tham khảo thủ công/khai báo danh mục sản phẩm mẫu, không thu thập dữ liệu cá nhân."
    },
    "fptshop": {
        "url": "https://fptshop.com.vn/",
        "mo_ta": "Tham khảo nhóm hàng công nghệ, laptop, điện thoại, phụ kiện.",
        "cach_dung": "Tham khảo thủ công/khai báo danh mục sản phẩm mẫu."
    },
    "cellphones": {
        "url": "https://cellphones.com.vn/",
        "mo_ta": "Tham khảo nhóm sản phẩm điện tử, linh kiện, phụ kiện và khoảng giá thị trường.",
        "cach_dung": "Tham khảo thủ công/khai báo danh mục sản phẩm mẫu."
    },
    "ghn": {
        "url": "https://ghn.vn/",
        "mo_ta": "Tham khảo đơn vị vận chuyển và nguyên tắc phí giao hàng theo khu vực/khoảng cách.",
        "cach_dung": "Dùng tên đơn vị vận chuyển và mô phỏng phí theo vùng."
    },
    "ghtk": {
        "url": "https://ghtk.vn/",
        "mo_ta": "Tham khảo đơn vị vận chuyển và nguyên tắc phí giao hàng nội tỉnh/liên tỉnh.",
        "cach_dung": "Dùng tên đơn vị vận chuyển và mô phỏng phí theo vùng."
    },
    "viettelpost": {
        "url": "https://viettelpost.com.vn/",
        "mo_ta": "Tham khảo đơn vị vận chuyển toàn quốc.",
        "cach_dung": "Dùng tên đơn vị vận chuyển và mô phỏng phí theo vùng."
    },
    "jtexpress": {
        "url": "https://jtexpress.vn/",
        "mo_ta": "Tham khảo đơn vị vận chuyển TMĐT.",
        "cach_dung": "Dùng tên đơn vị vận chuyển và mô phỏng phí theo vùng."
    },
    "dangkykinhdoanh": {
        "url": "https://dangkykinhdoanh.gov.vn/",
        "mo_ta": "Tham khảo bối cảnh doanh nghiệp/kinh doanh tại Việt Nam cho phần mô tả báo cáo.",
        "cach_dung": "Chỉ dùng làm nguồn tham khảo bối cảnh, không lấy dữ liệu cá nhân."
    }
}

RANDOM_SEED = 42
N_ORDERS = 968
OUTPUT_FILE = "BoDuLieu.xlsx"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def fetch_provinces_vietnam():
    """
    Lấy danh sách tỉnh/thành từ Province Open API.
    Nếu API lỗi hoặc mất mạng, dùng danh sách dự phòng để code vẫn chạy được.
    """
    url = SOURCES["hanh_chinh_vn"]["url"]
    fallback = [
        {"name": "TP. Hồ Chí Minh"}, {"name": "Cần Thơ"}, {"name": "Đồng Nai"},
        {"name": "Tây Ninh"}, {"name": "Vĩnh Long"}, {"name": "Đồng Tháp"},
        {"name": "An Giang"}, {"name": "Cà Mau"}, {"name": "Lâm Đồng"},
        {"name": "Khánh Hòa"}, {"name": "Đắc Lắk"}
    ]
    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        data = response.json()
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            return data.get("results") or data.get("data") or fallback
        return fallback
    except Exception as exc:
        print(f"Không lấy được API tỉnh/thành, dùng dữ liệu dự phòng. Lỗi: {exc}")
        return fallback


def normalize_province_name(item):
    """Chuẩn hóa tên tỉnh/thành từ dữ liệu API."""
    if isinstance(item, dict):
        return item.get("name") or item.get("full_name") or item.get("province_name") or "Không rõ"
    return str(item)


PRODUCTS = [
    {"Ma_San_Pham": "SP001", "Ten_San_Pham": "Samsung Galaxy A15", "thuong_hieu": "Samsung", "danh_muc": "Điện thoại", "Don_Gia": 4590000},
    {"Ma_San_Pham": "SP002", "Ten_San_Pham": "iPhone 15", "thuong_hieu": "Apple", "danh_muc": "Điện thoại", "Don_Gia": 18990000},
    {"Ma_San_Pham": "SP003", "Ten_San_Pham": "Xiaomi Redmi Note", "thuong_hieu": "Xiaomi", "danh_muc": "Điện thoại", "Don_Gia": 4990000},
    {"Ma_San_Pham": "SP004", "Ten_San_Pham": "Laptop ASUS Vivobook", "thuong_hieu": "ASUS", "danh_muc": "Laptop", "Don_Gia": 13990000},
    {"Ma_San_Pham": "SP005", "Ten_San_Pham": "Laptop Dell Inspiron", "thuong_hieu": "Dell", "danh_muc": "Laptop", "Don_Gia": 15990000},
    {"Ma_San_Pham": "SP006", "Ten_San_Pham": "Laptop HP Pavilion", "thuong_hieu": "HP", "danh_muc": "Laptop", "Don_Gia": 14990000},
    {"Ma_San_Pham": "SP007", "Ten_San_Pham": "Chuột Logitech M331", "thuong_hieu": "Logitech", "danh_muc": "Phụ kiện", "Don_Gia": 329000},
    {"Ma_San_Pham": "SP008", "Ten_San_Pham": "Bàn phím cơ AKKO", "thuong_hieu": "AKKO", "danh_muc": "Phụ kiện", "Don_Gia": 1490000},
    {"Ma_San_Pham": "SP009", "Ten_San_Pham": "Ổ cứng SSD Kingston 500GB", "thuong_hieu": "Kingston", "danh_muc": "Linh kiện", "Don_Gia": 990000},
    {"Ma_San_Pham": "SP010", "Ten_San_Pham": "RAM DDR4 8GB", "thuong_hieu": "Kingston", "danh_muc": "Linh kiện", "Don_Gia": 650000},
    {"Ma_San_Pham": "SP011", "Ten_San_Pham": "Tai nghe AirPods 3", "thuong_hieu": "Apple", "danh_muc": "Tai nghe", "Don_Gia": 3990000},
    {"Ma_San_Pham": "SP012", "Ten_San_Pham": "Tai nghe Sony WH-CH520", "thuong_hieu": "Sony", "danh_muc": "Tai nghe", "Don_Gia": 1290000},
    {"Ma_San_Pham": "SP013", "Ten_San_Pham": "Màn hình LG 24 inch", "thuong_hieu": "LG", "danh_muc": "Màn hình", "Don_Gia": 2990000},
    {"Ma_San_Pham": "SP014", "Ten_San_Pham": "Màn hình Samsung 27 inch", "thuong_hieu": "Samsung", "danh_muc": "Màn hình", "Don_Gia": 4290000},
    {"Ma_San_Pham": "SP015", "Ten_San_Pham": "Máy in Canon LBP 2900", "thuong_hieu": "Canon", "danh_muc": "Máy in", "Don_Gia": 4290000},
]

FIRST_NAMES = ["An", "Bình", "Châu", "Dũng", "Giang", "Hạnh", "Khang", "Linh", "Minh", "Nam", "Phúc", "Quân", "Tâm", "Trang", "Trúc", "Vy"]
LAST_NAMES = ["Nguyễn", "Trần", "Lê", "Phạm", "Hoàng", "Huỳnh", "Võ", "Đặng", "Bùi", "Đỗ", "Ngô", "Lý", "Tạ"]
MIDDLE_NAMES = ["Văn", "Thị", "Minh", "Gia", "Bảo", "Đức", "Thanh", "Ngọc"]
STREETS = ["Nguyễn Trãi", "Lê Lợi", "Trần Hưng Đạo", "Cách Mạng Tháng Tám", "Pasteur", "Võ Văn Kiệt", "Phạm Văn Đồng"]
WARDS = ["Phường Trung Tâm", "Phường Bình Thạnh", "Phường Ninh Kiều", "Phường Cái Răng", "Phường Cà Mau", "Phường Trảng Bàng", "Phường Biên Hòa"]
CARRIERS = ["Giao Hàng Nhanh", "Giao Hàng Tiết Kiệm", "Viettel Post", "VNPost", "J&T Express"]
PAYMENTS = ["COD", "Chuyển khoản", "Ví điện tử", "Thẻ ngân hàng"]
CHANNELS = ["Shopee", "Lazada", "TikTok Shop", "Facebook", "Website", "Zalo"]
STATUSES = ["Đã giao", "Đang giao", "Đã hủy", "Hoàn hàng"]


def random_phone():
    return "0" + random.choice(["90", "91", "93", "94", "96", "97", "98", "86", "88"]) + "".join(random.choices(string.digits, k=7))


def random_customer_name():
    gender_title = random.choice(["Anh", "Chị"])
    name = f"{random.choice(LAST_NAMES)} {random.choice(MIDDLE_NAMES)} {random.choice(FIRST_NAMES)}"
    return gender_title, name


def shipping_fee(province_name, quantity, product_category):
    near = ["Hồ Chí Minh", "Cần Thơ", "Đồng Nai", "Bình Dương"]
    far = ["Lâm Đồng", "Đắc Lắc", "Khánh Hòa", "Cà Mau", "An Giang"]

    if any(x in province_name for x in near):
        base = random.randint(18000, 28000)
    elif any(x in province_name for x in far):
        base = random.randint(35000, 55000)
    else:
        base = random.randint(28000, 42000)

    if product_category in ["Laptop", "Màn hình", "Máy in"]:
        base += random.randint(10000, 25000)

    base += max(quantity - 1, 0) * random.randint(3000, 7000)
    return int(round(base / 1000) * 1000)


def expected_delivery_date(order_date, province_name):
    if any(x in province_name for x in ["Hồ Chí Minh", "Cần Thơ", "Đồng Nai", "Bình Dương"]):
        days = random.randint(1, 3)
    elif any(x in province_name for x in ["Lâm Đồng", "Đắc Lắc", "Khánh Hòa", "Cà Mau", "An Giang"]):
        days = random.randint(4, 7)
    else:
        days = random.randint(2, 5)
    return order_date + timedelta(days=days)


def generate_orders(n_orders=N_ORDERS):
    provinces_raw = fetch_provinces_vietnam()
    province_names = [normalize_province_name(x) for x in provinces_raw]
    province_names = [x for x in province_names if x and x != "Không rõ"]

    preferred_keywords = ["Hồ Chí Minh", "Cần Thơ", "Đồng Nai", "Tây Ninh", "Vĩnh Long", "Đồng Tháp", "An Giang", "Cà Mau", "Lâm Đồng", "Khánh Hòa", "Đắc Lắc"]
    preferred = [p for p in province_names if any(k in p for k in preferred_keywords)]
    if len(preferred) >= 5:
        province_names = preferred

    start_date = datetime(2020, 1, 1)
    end_date = datetime(2025, 12, 31)
    total_days = (end_date - start_date).days

    rows = []
    for i in range(1, n_orders + 1):
        order_date = start_date + timedelta(days=random.randint(0, total_days))
        product = random.choice(PRODUCTS).copy()
        quantity = int(np.random.choice([1, 1, 1, 2, 2, 3, 4], p=[0.35, 0.2, 0.1, 0.18, 0.07, 0.06, 0.04]))
        discount_rate = float(np.random.choice([0, 0.05, 0.10, 0.15, 0.20], p=[0.45, 0.18, 0.18, 0.12, 0.07]))

        province = random.choice(province_names)
        gender_title, customer_name = random_customer_name()
        gross = product["Don_Gia"] * quantity
        discount = int(round(gross * discount_rate / 1000) * 1000)
        taxable = gross - discount
        vat = int(round(taxable * 0.10 / 1000) * 1000)
        ship = shipping_fee(province, quantity, product["danh_muc"])
        total = taxable + vat + ship

        row = {
            "Ma_Don": f"DH{order_date.year}{i:06d}",
            "Ngay_Dat": order_date.date(),
            "Ma_Khach_Hang": f"KH{random.randint(1, 1200):05d}",
            "Ten_Khach_Hang": customer_name,
            "Gioi_Xung_Ho": gender_title,
            "So_Dien_Thoai": random_phone(),
            "Tinh_Thanh": province,
            "Phuong_Xa": random.choice(WARDS),
            "Dia_Chi_Cu_The": f"{random.randint(1, 999)} {random.choice(STREETS)}",
            "Ma_San_Pham": product["Ma_San_Pham"],
            "Ten_San_Pham": product["Ten_San_Pham"],
            "Nhom_Hang": "Đồ điện tử",
            "So_Luong": quantity,
            "Don_Gia": product["Don_Gia"],
            "Ty_Le_Giam_Gia": discount_rate,
            "Tien_Giam": discount,
            "Tong_Hang": gross,
            "VAT_10": vat,
            "Phi_Van_Chuyen": ship,
            "Tong_Thanh_Toan": total,
            "Don_Vi_Giao": random.choice(CARRIERS),
            "Phuong_Thuc_Thanh_Toan": random.choice(PAYMENTS),
            "Kenh_Ban": random.choice(CHANNELS),
            "Trang_Thai": random.choices(STATUSES, weights=[0.78, 0.12, 0.05, 0.05])[0],
            "Ngay_Giao_Du_Kien": expected_delivery_date(order_date, province).date(),
            "So_Lan_Khach_Quay_Lai": int(np.random.poisson(5)),
            "thuong_hieu": product["thuong_hieu"],
            "danh_muc": product["danh_muc"],
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    df = df.sort_values("Ngay_Dat").reset_index(drop=True)
    return df


def build_source_sheet():
    rows = []
    rows.append({
        "Mục": "Loại dữ liệu",
        "Ghi chú": "Bộ dữ liệu đơn hàng TMĐT linh kiện/điện tử mô phỏng thực tế; không chứa thông tin cá nhân thật."
    })
    rows.append({
        "Mục": "Phạm vi thời gian",
        "Ghi chú": "Tạo đơn hàng trong giai đoạn 2020-01-01 đến 2025-12-31."
    })
    rows.append({
        "Mục": "Nguồn địa giới",
        "Ghi chú": SOURCES["hanh_chinh_vn"]["url"]
    })
    for key, value in SOURCES.items():
        rows.append({
            "Mục": f"Nguồn tham khảo - {key}",
            "Ghi chú": f"{value['url']} | {value['mo_ta']} | Cách dùng: {value['cach_dung']}"
        })
    rows.append({
        "Mục": "Ngày tạo",
        "Ghi chú": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })
    return pd.DataFrame(rows)


def build_overview_sheet(df):
    overview = [
        ["Chỉ số", "Giá trị"],
        ["Số dòng dữ liệu", len(df)],
        ["Số khách hàng", df["Ma_Khach_Hang"].nunique()],
        ["Số sản phẩm", df["Ma_San_Pham"].nunique()],
        ["Số tỉnh/thành", df["Tinh_Thanh"].nunique()],
        ["Tổng doanh thu", int(df["Tong_Thanh_Toan"].sum())],
        ["Doanh thu trung bình/đơn", int(df["Tong_Thanh_Toan"].mean())],
        ["Phí vận chuyển trung bình", int(df["Phi_Van_Chuyen"].mean())],
        ["Từ ngày", str(df["Ngay_Dat"].min())],
        ["Đến ngày", str(df["Ngay_Dat"].max())],
    ]
    return pd.DataFrame(overview[1:], columns=overview[0])


def export_excel(df, output_file=OUTPUT_FILE):
    source_df = build_source_sheet()
    overview_df = build_overview_sheet(df)

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="DuLieu_10000", index=False)
        source_df.to_excel(writer, sheet_name="Nguon_GhiChu", index=False)
        overview_df.to_excel(writer, sheet_name="TongQuan", index=False)

        workbook = writer.book
        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            ws.freeze_panes = "A2"
            for col in ws.columns:
                max_len = 0
                col_letter = col[0].column_letter
                for cell in col:
                    text = "" if cell.value is None else str(cell.value)
                    max_len = max(max_len, len(text))
                ws.column_dimensions[col_letter].width = min(max_len + 2, 45)

    print(f"Đã xuất file: {Path(output_file).resolve()}")


if __name__ == "__main__":
    df_orders = generate_orders(N_ORDERS)
    export_excel(df_orders, OUTPUT_FILE)